# Customer-Level Feature Engineering

This notebook builds a customer-level analytical dataset for the Olist customer segmentation project.

The raw Olist dataset is relational: customers, orders, items, payments, reviews, and products are stored in separate tables. K-Means clustering needs one row per customer, so this notebook aggregates order-level, payment-level, item-level, and review-level information into customer-level behavior features.

This notebook does not run K-Means yet.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_data_dir = project_root / "data" / "raw"
processed_data_dir = project_root / "data" / "processed"
processed_data_dir.mkdir(parents=True, exist_ok=True)


def relative_project_path(path):
    return Path(path).resolve().relative_to(project_root).as_posix()


print("Project root: repository root resolved")
print("Raw data directory:", relative_project_path(raw_data_dir))
print("Raw data directory exists:", raw_data_dir.exists())
print("Processed data directory:", relative_project_path(processed_data_dir))
print("Processed data directory exists:", processed_data_dir.exists())

Project root: repository root resolved
Raw data directory: data/raw
Raw data directory exists: True
Processed data directory: data/processed
Processed data directory exists: True


In [3]:
customers = pd.read_csv(raw_data_dir / "olist_customers_dataset.csv")
orders = pd.read_csv(raw_data_dir / "olist_orders_dataset.csv")
order_items = pd.read_csv(raw_data_dir / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(raw_data_dir / "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(raw_data_dir / "olist_order_reviews_dataset.csv")
products = pd.read_csv(raw_data_dir / "olist_products_dataset.csv")
category_translation = pd.read_csv(raw_data_dir / "product_category_name_translation.csv")

tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products,
    "category_translation": category_translation,
}

for table_name, df in tables.items():
    print(f"{table_name}: {df.shape[0]:,} rows, {df.shape[1]:,} columns")

customers: 99,441 rows, 5 columns
orders: 99,441 rows, 8 columns
order_items: 112,650 rows, 7 columns
order_payments: 103,886 rows, 5 columns
order_reviews: 99,224 rows, 7 columns
products: 32,951 rows, 9 columns
category_translation: 71 rows, 2 columns


## Timestamp Conversion

Timestamp columns are read from CSV as object/string values. Before calculating recency, delivery duration, or late delivery indicators, the columns must be converted explicitly into datetime.

In [4]:
orders_clean = orders.copy()

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in date_columns:
    orders_clean[column] = pd.to_datetime(orders_clean[column], errors="coerce")

display(orders_clean[date_columns].dtypes)

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## Completed Purchase Scope

The main customer segmentation dataset focuses on delivered orders. Delivered orders represent completed purchases, which are more reliable for purchase behavior, payment value, review score, and delivery experience features.

Non-delivered orders are not used in the base customer feature table because canceled, unavailable, shipped, processing, or created orders may not represent completed customer purchases.

In [5]:
delivered_orders = orders_clean.loc[orders_clean["order_status"].eq("delivered")].copy()

valid_delivered_orders = delivered_orders.dropna(
    subset=[
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ]
).copy()

scope_summary = pd.DataFrame([
    {"metric": "all_orders", "value": len(orders_clean)},
    {"metric": "delivered_orders", "value": len(delivered_orders)},
    {"metric": "delivered_orders_missing_delivery_date", "value": int(delivered_orders["order_delivered_customer_date"].isna().sum())},
    {"metric": "valid_delivered_orders_for_features", "value": len(valid_delivered_orders)},
])

display(scope_summary)

,metric,value
0,all_orders,99441
1,delivered_orders,96478
2,delivered_orders_missing_delivery_date,8
3,valid_delivered_orders_for_features,96470


In [6]:
orders_with_customers = valid_delivered_orders.merge(
    customers[["customer_id", "customer_unique_id", "customer_city", "customer_state"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
)

print("Orders with customers shape:", orders_with_customers.shape)
print("Missing customer_unique_id:", orders_with_customers["customer_unique_id"].isna().sum())
display(orders_with_customers.head())

Orders with customers shape: (96470, 11)
Missing customer_unique_id: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP


## Payment Aggregation

The payment table can have more than one row per order because an order can be paid with multiple payment records. Therefore, payment features must be aggregated to order level before joining with orders.

In [7]:
payment_order_features = (
    order_payments
    .groupby("order_id")
    .agg(
        order_payment_value=("payment_value", "sum"),
        order_payment_installments_mean=("payment_installments", "mean"),
        order_payment_method_count=("payment_type", "nunique"),
    )
    .reset_index()
)

print("Payment order features shape:", payment_order_features.shape)
display(payment_order_features.head())

Payment order features shape: (99440, 4)


,order_id,order_payment_value,order_payment_installments_mean,order_payment_method_count
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2.00,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,3.00,1
2,000229ec398224ef6ca0657da4fc703e,216.87,5.00,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,2.00,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3.00,1


## Review Aggregation

The review table may contain more than one review row for some orders. The base feature table uses average review score per order and review count as order-level review features.

In [8]:
review_order_features = (
    order_reviews
    .groupby("order_id")
    .agg(
        order_review_score_mean=("review_score", "mean"),
        order_review_count=("review_id", "nunique"),
    )
    .reset_index()
)

print("Review order features shape:", review_order_features.shape)
display(review_order_features.head())

Review order features shape: (98673, 3)


,order_id,order_review_score_mean,order_review_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.00,1
1,00018f77f2f0320c557190d7a144bdd3,4.00,1
2,000229ec398224ef6ca0657da4fc703e,5.00,1
3,00024acbcdf0a6daa1e931b038114c75,4.00,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.00,1


## Item and Product Aggregation

The order items table is item-level, so one order can appear multiple times. Product categories are attached through the products table. Category translation is used when available, and the original category name is used as fallback if translation is missing.

In [9]:
products_with_category = products.merge(
    category_translation,
    on="product_category_name",
    how="left",
)

products_with_category["product_category_name_english"] = (
    products_with_category["product_category_name_english"]
    .fillna(products_with_category["product_category_name"])
)

items_with_products = order_items.merge(
    products_with_category[[
        "product_id",
        "product_category_name",
        "product_category_name_english",
    ]],
    on="product_id",
    how="left",
    validate="many_to_one",
)

item_order_features = (
    items_with_products
    .groupby("order_id")
    .agg(
        order_item_count=("order_item_id", "count"),
        order_product_count=("product_id", "nunique"),
        order_category_count=("product_category_name_english", "nunique"),
        order_item_price_sum=("price", "sum"),
        order_freight_value_sum=("freight_value", "sum"),
    )
    .reset_index()
)

print("Item order features shape:", item_order_features.shape)
display(item_order_features.head())

Item order features shape: (98666, 6)


,order_id,order_item_count,order_product_count,order_category_count,order_item_price_sum,order_freight_value_sum
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14


In [10]:
order_features = (
    orders_with_customers
    .merge(payment_order_features, on="order_id", how="left", validate="one_to_one")
    .merge(review_order_features, on="order_id", how="left", validate="one_to_one")
    .merge(item_order_features, on="order_id", how="left", validate="one_to_one")
)

join_quality = pd.DataFrame([
    {"check": "missing_payment_value", "count": int(order_features["order_payment_value"].isna().sum())},
    {"check": "missing_review_score", "count": int(order_features["order_review_score_mean"].isna().sum())},
    {"check": "missing_item_count", "count": int(order_features["order_item_count"].isna().sum())},
])

print("Order features shape:", order_features.shape)
display(join_quality)

Order features shape: (96470, 21)


,check,count
0,missing_payment_value,1
1,missing_review_score,646
2,missing_item_count,0


In [11]:
order_features["delivery_days"] = (
    order_features["order_delivered_customer_date"] - order_features["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

order_features["approval_days"] = (
    order_features["order_approved_at"] - order_features["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

order_features["is_late_delivery"] = (
    order_features["order_delivered_customer_date"] > order_features["order_estimated_delivery_date"]
).astype(int)

reference_date = order_features["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

print("Reference date:", reference_date)
display(order_features[[
    "order_id",
    "customer_unique_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_days",
    "approval_days",
    "is_late_delivery",
]].head())

Reference date: 2018-08-30 15:00:37


,order_id,customer_unique_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,approval_days,is_late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.44,0.01,0
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.78,1.28,0
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.39,0.01,0
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13.21,0.01,0
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2.87,0.04,0


## Customer-Level Aggregation

This section aggregates order-level behavior into customer-level features. The final row level is `customer_unique_id`, because customer segmentation should represent real customers, not individual orders.

In [12]:
customer_features = (
    order_features
    .groupby("customer_unique_id")
    .agg(
        first_purchase_date=("order_purchase_timestamp", "min"),
        last_purchase_date=("order_purchase_timestamp", "max"),
        order_count=("order_id", "nunique"),
        total_payment_value=("order_payment_value", "sum"),
        avg_payment_value=("order_payment_value", "mean"),
        avg_payment_installments=("order_payment_installments_mean", "mean"),
        avg_payment_method_count=("order_payment_method_count", "mean"),
        total_items=("order_item_count", "sum"),
        avg_items_per_order=("order_item_count", "mean"),
        total_product_count=("order_product_count", "sum"),
        avg_product_count_per_order=("order_product_count", "mean"),
        max_category_count_per_order=("order_category_count", "max"),
        total_item_price=("order_item_price_sum", "sum"),
        total_freight_value=("order_freight_value_sum", "sum"),
        avg_freight_value=("order_freight_value_sum", "mean"),
        avg_review_score=("order_review_score_mean", "mean"),
        review_count=("order_review_count", "sum"),
        avg_delivery_days=("delivery_days", "mean"),
        late_delivery_rate=("is_late_delivery", "mean"),
    )
    .reset_index()
)

customer_features["recency_days"] = (
    reference_date - customer_features["last_purchase_date"]
).dt.total_seconds() / (60 * 60 * 24)

customer_features["customer_lifetime_days"] = (
    customer_features["last_purchase_date"] - customer_features["first_purchase_date"]
).dt.total_seconds() / (60 * 60 * 24)

customer_features["freight_to_payment_ratio"] = (
    customer_features["total_freight_value"] / customer_features["total_payment_value"].replace(0, np.nan)
)

customer_features = customer_features[
    [
        "customer_unique_id",
        "recency_days",
        "customer_lifetime_days",
        "order_count",
        "total_payment_value",
        "avg_payment_value",
        "avg_payment_installments",
        "avg_payment_method_count",
        "total_items",
        "avg_items_per_order",
        "total_product_count",
        "avg_product_count_per_order",
        "max_category_count_per_order",
        "total_item_price",
        "total_freight_value",
        "avg_freight_value",
        "freight_to_payment_ratio",
        "avg_review_score",
        "review_count",
        "avg_delivery_days",
        "late_delivery_rate",
        "first_purchase_date",
        "last_purchase_date",
    ]
]

print("Customer features shape:", customer_features.shape)
display(customer_features.head())

Customer features shape: (93350, 23)


,customer_unique_id,recency_days,customer_lifetime_days,order_count,total_payment_value,avg_payment_value,avg_payment_installments,avg_payment_method_count,total_items,avg_items_per_order,total_product_count,avg_product_count_per_order,max_category_count_per_order,total_item_price,total_freight_value,avg_freight_value,freight_to_payment_ratio,avg_review_score,review_count,avg_delivery_days,late_delivery_rate,first_purchase_date,last_purchase_date
0,0000366f3b9a7992bf8c76cfdf3221e2,112.17,0.00,1,141.90,141.90,8.00,1.00,1,1.00,1,1.00,1,129.90,12.00,12.00,0.08,5.00,1.00,6.41,0.00,2018-05-10 10:56:27,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115.16,0.00,1,27.19,27.19,1.00,1.00,1,1.00,1,1.00,1,18.90,8.29,8.29,0.30,4.00,1.00,3.29,0.00,2018-05-07 11:11:27,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,537.75,0.00,1,86.22,86.22,8.00,1.00,1,1.00,1,1.00,1,69.00,17.22,17.22,0.20,3.00,1.00,25.73,0.00,2017-03-10 21:05:03,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,321.77,0.00,1,43.62,43.62,4.00,1.00,1,1.00,1,1.00,1,25.99,17.63,17.63,0.40,4.00,1.00,20.04,0.00,2017-10-12 20:29:41,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,288.80,0.00,1,196.89,196.89,6.00,1.00,1,1.00,1,1.00,1,180.00,16.89,16.89,0.09,5.00,1.00,13.14,0.00,2017-11-14 19:45:42,2017-11-14 19:45:42


In [13]:
feature_quality = pd.DataFrame([
    {"check": "rows", "value": len(customer_features)},
    {"check": "unique_customer_unique_id", "value": customer_features["customer_unique_id"].nunique()},
    {"check": "duplicate_customer_unique_id", "value": int(customer_features["customer_unique_id"].duplicated().sum())},
    {"check": "missing_avg_payment_value", "value": int(customer_features["avg_payment_value"].isna().sum())},
    {"check": "missing_avg_review_score", "value": int(customer_features["avg_review_score"].isna().sum())},
    {"check": "missing_freight_to_payment_ratio", "value": int(customer_features["freight_to_payment_ratio"].isna().sum())},
])

display(feature_quality)

,check,value
0,rows,93350
1,unique_customer_unique_id,93350
2,duplicate_customer_unique_id,0
3,missing_avg_payment_value,1
4,missing_avg_review_score,603
5,missing_freight_to_payment_ratio,1


In [14]:
summary_columns = [
    "recency_days",
    "order_count",
    "total_payment_value",
    "avg_payment_value",
    "avg_review_score",
    "avg_delivery_days",
    "late_delivery_rate",
]

display(customer_features[summary_columns].describe())

,recency_days,order_count,total_payment_value,avg_payment_value,avg_review_score,avg_delivery_days,late_delivery_rate
count,"93,350.00","93,350.00","93,350.00","93,349.00","92,747.00","93,350.00","93,350.00"
mean,238.48,1.03,165.20,160.32,4.15,12.57,0.08
std,152.59,0.21,226.32,219.58,1.28,9.55,0.27
min,1.00,1.00,0.00,9.59,1.00,0.53,0.00
25%,114.92,1.00,63.05,62.37,4.00,6.79,0.00
50%,219.63,1.00,107.78,105.63,5.00,10.23,0.00
75%,346.86,1.00,182.55,176.65,5.00,15.72,0.00
max,714.11,15.00,"13,664.08","13,664.08",5.00,209.63,1.00


## Save Customer-Level Dataset

The customer-level feature table is saved locally to `data/processed/customer_features_base.csv`. This file is ignored by Git and should not be committed.

In [15]:
output_path = processed_data_dir / "customer_features_base.csv"
customer_features.to_csv(output_path, index=False)

print("Saved to:", relative_project_path(output_path))
print("Rows:", len(customer_features))
print("Columns:", customer_features.shape[1])

Saved to: data/processed/customer_features_base.csv
Rows: 93350
Columns: 23


## Customer-Level Aggregation Summary

This notebook created a customer-level feature table from delivered Olist orders.

**Important decisions:**
* The base dataset focuses on `delivered` orders because they represent completed purchases.
* Timestamp columns were converted explicitly before calculating time-based features.
* Payment, review, item, and product information were aggregated to the order level before customer-level aggregation.
* The final row level is `customer_unique_id`.
* Geolocation and seller tables are not included in the base feature set because they are optional and not required for the core customer purchase behavior segmentation.
* K-Means is not run in this notebook. Modeling will be handled after feature validation and preprocessing.

**Known issues for the next steps:**
* Some customers have missing review scores.
* One customer/order path has missing payment-derived values.
* Monetary and freight-related features are likely skewed.
* Feature selection and preprocessing are still needed before K-Means.